# Seed Event Hubs

Publish JSONL fixtures to Azure Event Hubs topics using the Kafka protocol.
Requires a valid bootstrap server and JAAS config (Databricks secret).

In [ ]:
dbutils.widgets.text("source_path", "")
dbutils.widgets.text("eh_bootstrap_servers", "")
dbutils.widgets.text("eh_claims_topic", "actuarial.claims")
dbutils.widgets.text("eh_premiums_topic", "actuarial.premiums")
dbutils.widgets.text("eh_risk_zones_topic", "actuarial.risk_zones")
dbutils.widgets.text("eh_cyclone_events_topic", "actuarial.cyclone_events")
dbutils.widgets.text("eh_jaas_config", "")

source_path = dbutils.widgets.get("source_path").rstrip("/")
bootstrap = dbutils.widgets.get("eh_bootstrap_servers")
jaas = dbutils.widgets.get("eh_jaas_config")

datasets = [
    ("claims", dbutils.widgets.get("eh_claims_topic")),
    ("premiums", dbutils.widgets.get("eh_premiums_topic")),
    ("risk_zones", dbutils.widgets.get("eh_risk_zones_topic")),
    ("cyclone_events", dbutils.widgets.get("eh_cyclone_events_topic")),
]

print(f"source_path={source_path}")
print(f"bootstrap={bootstrap}")
for subdir, topic in datasets:
    print(f"  {subdir} -> {topic}")

In [ ]:
from pyspark.sql import functions as F


def publish_jsonl(subdir: str, topic: str) -> int:
    path = f"{source_path}/{subdir}"
    df = (
        spark.read.text(path)
        .filter(F.col("value").isNotNull() & (F.length(F.trim(F.col("value"))) > 0))
        .select(F.col("value").cast("string").alias("value"))
    )
    count = df.count()
    if count == 0:
        print(f"SKIP {subdir}: no lines under {path}")
        return 0

    (
        df.write.format("kafka")
        .option("kafka.bootstrap.servers", bootstrap)
        .option("topic", topic)
        .option("kafka.security.protocol", "SASL_SSL")
        .option("kafka.sasl.mechanism", "PLAIN")
        .option("kafka.sasl.jaas.config", jaas)
        .save()
    )
    print(f"Published {count} messages to {topic} from {path}")
    return count


if not bootstrap or not jaas or "YOUR_NAMESPACE" in bootstrap:
    raise ValueError(
        "Configure eh_bootstrap_servers and eh_jaas_config (secret) before seeding Event Hubs."
    )

total = 0
for subdir, topic in datasets:
    total += publish_jsonl(subdir, topic)
print(f"Done. Total messages published: {total}")